In [ ]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
from matplotlib.patches import Ellipse

In [ ]:
# Generate polytope as convex hull of p random points
np.random.seed(0)
n = 2
p = 50
x = np.random.randn(p, n)
hull = ConvexHull(x)

# Extract matrices of polytope
C = - hull.equations[:, :-1]
d = - hull.equations[:, -1]

In [ ]:
# Ellipsoid matrix and offset
A = cp.Variable((n, n), PSD=True)
b = cp.Variable(n)

# Semidefinite program
objective = - cp.log_det(A)
constraints = [cp.norm2(A @ ci) <= ci @ b + di for ci, di in zip(C, d)]
prob = cp.Problem(cp.Minimize(objective), constraints)
prob.solve()

In [ ]:
if n == 2:

    # Get ellipse axes and angle
    P = np.linalg.inv(A.value.T @ A.value)
    eigvals, eigvecs = np.linalg.eigh(P)
    axes = 2 / np.sqrt(eigvals)
    angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))

    # Plot ellipse and polytope
    plt.figure()
    plt.axis('equal')
    plt.gca().add_patch(Ellipse(b.value, axes[0], axes[1], angle=angle, ec='k', fc='None'))
    vertices = x[hull.vertices]
    closed_vertices = np.vstack([vertices, vertices[0]])
    plt.plot(*closed_vertices.T)
    plt.show()